# Permutation test — tín hiệu giai đoạn trên mẫu u (RQ0)

Kiểm tra biểu hiện gen của **mẫu u** có mang tín hiệu phân biệt giai đoạn hay không (Review 04/10/2026: R1, R2, mục 8.6.2; trả lời **RQ0**).

- **Dữ liệu:** gộp `train/val/test` của `fold_1` (= toàn bộ bộ dữ liệu), **bỏ mẫu Normal**. Giữ nguyên log2 expression, không chuẩn hoá.
- **Chia fold:** `StratifiedGroupKFold(N_SPLITS)` theo **bệnh nhân** (TCGA: 12 ký tự đầu barcode), có assert không trùng bệnh nhân giữa train/test.
- **Pipeline (mỗi fold):** F-test chọn top `N_FEATURES` gen **chỉ trên phần train của fold** → `LogisticRegression(class_weight="balanced")`. Bước chọn gen nằm *trong* vòng CV nên được chạy lại ở mọi fold và mọi lần hoán vị.
- **Nhiệm vụ:** `stage_all` (I/II/III/IV), `early_late` (I vs II–IV), `stage_merge_34` (I/II/III+IV, chỉ khi có lớp IV).
- **Ba phép so sánh** (chỉ số: BAcc trung bình qua các fold):
  1. **Nhãn thật** — pipeline trên.
  2. **Permutation:** xáo nhãn `N_PERMUTATIONS` lần (theo bệnh nhân), chạy lại *toàn bộ* pipeline → phân phối null; p = (#null ≥ thật + 1)/(N + 1).
  3. **Random-k:** `N_FEATURES` gen ngẫu nhiên + cùng bộ phân loại, `N_RANDOM_K` lần; và Dummy (stratified).
- **Tiêu chí RQ0 (đặt trước):** đạt nếu p_perm < 0,05 **và** tỉ lệ lần random-k ≥ nhãn thật < 0,05.
- **Đổi dataset:** sửa `DATA_DIR` ở cell cấu hình.
- **Đầu ra** (`/kaggle/working/perm_test/<dataset>/`): `summary.csv`, `null_scores.csv`, `random_k_scores.csv`, `real_per_fold.csv`, `real_confusion_<task>.csv`, `perm_test.png`, `config.json` → nén `perm_test_<dataset>.zip`.

In [ ]:
# tables: backend cho pd.read_hdf
get_ipython().system("pip install -q tables")

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
# Doi DATA_DIR sang bo khac (vd .../h5/tcga_luad, .../h5/tcga_lusc) de chay dataset khac
DATA_DIR = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5/gse68465"
N_FEATURES = 50           # so gene F-test giu lai trong moi fold
N_SPLITS = 5
N_PERMUTATIONS = 200      # so lan xao nhan (p nho nhat = 1/201)
N_RANDOM_K = 100          # so lan lap baseline random-k
SEED = 42
N_JOBS = -1
LR_PARAMS = {"C": 1.0, "class_weight": "balanced", "max_iter": 5000}
TASKS = ["stage_all", "early_late", "stage_merge_34"]
ALPHA = 0.05

DATASET_NAME = os.path.basename(os.path.normpath(DATA_DIR))
RESULT_DIR = os.path.join("/kaggle/working/perm_test", DATASET_NAME)
os.makedirs(RESULT_DIR, exist_ok=True)

assert os.path.isfile(os.path.join(DATA_DIR, "fold_1", "train.h5")), \
    f"Khong thay {DATA_DIR}/fold_1/train.h5 -- da Add Input dataset chua?"
print("Dataset :", DATASET_NAME, "->", DATA_DIR)
print("Ket qua :", RESULT_DIR)
print("So CPU  :", os.cpu_count())

## Đọc dữ liệu (chỉ mẫu u)

In [ ]:
import json, time, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}


def load_all():
    # train/val/test cua fold_1 la 3 phan roi nhau, gop lai = toan bo bo du lieu
    parts = [pd.read_hdf(os.path.join(DATA_DIR, "fold_1", f"{s}.h5"), key="data") for s in ["train", "val", "test"]]
    df = pd.concat(parts)
    assert df.index.is_unique, "Trung sample_id giua train/val/test"
    return df


df = load_all()
print(f"Toan bo: {df.shape[0]} mau x {df.shape[1] - 1} gene")
print("Phan bo lop (truoc khi bo Normal):", df["label"].map(STAGE_NAMES).value_counts().to_dict())

df = df[df["label"] != 0]                                   # BO mau Normal
X = df.drop(columns="label").values.astype(np.float64)      # giu nguyen log2, KHONG chuan hoa
y_stage = df["label"].values.astype(int)
gene_names = df.drop(columns="label").columns.to_numpy()
sample_ids = df.index.to_numpy().astype(str)

# Ma benh nhan: TCGA = 12 ky tu dau barcode; bo khac: moi mau la 1 benh nhan
groups = np.array([s[:12] if s.startswith("TCGA-") else s for s in sample_ids])
n_dup = int((pd.Series(groups).value_counts() > 1).sum())
print(f"Mau u: {X.shape[0]} | benh nhan: {len(np.unique(groups))} | benh nhan co >1 mau: {n_dup}")
print("Phan bo giai doan:", pd.Series(y_stage).map(STAGE_NAMES).value_counts().sort_index().to_dict())

In [ ]:
# Nhan cho tung nhiem vu
TASK_LABELS = {
    "stage_all": y_stage,
    "early_late": (y_stage >= 2).astype(int),              # 0 = som (I), 1 = muon (II-IV)
    "stage_merge_34": np.minimum(y_stage, 3),               # gop IV vao III
}
if 4 not in y_stage:                                        # khong co lop IV -> merge_34 trung stage_all
    TASKS = [t for t in TASKS if t != "stage_merge_34"]
TASK_CHANCE = {t: 1 / len(np.unique(TASK_LABELS[t])) for t in TASKS}
for t in TASKS:
    print(f"{t:15s} lop = {np.unique(TASK_LABELS[t]).tolist()} | muc ngau nhien BAcc = {TASK_CHANCE[t]:.3f}")

## Chia fold theo bệnh nhân và pipeline

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef, confusion_matrix, recall_score

# Fold co dinh, phan tang theo giai doan that; dung chung cho moi nhiem vu va moi lan hoan vi
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
SPLITS = list(sgkf.split(X, y_stage, groups))
for i, (tr, te) in enumerate(SPLITS, 1):
    assert set(groups[tr]).isdisjoint(groups[te]), f"fold {i}: trung benh nhan train/test"
    print(f"fold {i}: train {len(tr)} | test {len(te)} | test giai doan "
          f"{pd.Series(y_stage[te]).map(STAGE_NAMES).value_counts().sort_index().to_dict()}")


def ftest_top_k(Xtr, ytr, k):
    F, _ = f_classif(Xtr, ytr)                     # gene hang so -> NaN, day xuong cuoi
    F = np.nan_to_num(F, nan=-np.inf)
    return np.argsort(-F, kind="stable")[:k]


def run_cv(y, selector="ftest", rng=None, model="lr"):
    # Chon gen + huan luyen CHI tren phan train cua tung fold; tra ve BAcc tung fold va du doan gop
    pred = np.empty_like(y)
    fold_bacc = []
    for tr, te in SPLITS:
        if selector == "ftest":
            idx = ftest_top_k(X[tr], y[tr], N_FEATURES)
        else:
            idx = rng.choice(X.shape[1], size=N_FEATURES, replace=False)
        if model == "lr":
            clf = LogisticRegression(**LR_PARAMS)
        else:
            clf = DummyClassifier(strategy="stratified", random_state=int(rng.integers(1 << 31)))
        clf.fit(X[tr][:, idx], y[tr])
        pred[te] = clf.predict(X[te][:, idx])
        fold_bacc.append(balanced_accuracy_score(y[te], pred[te]))
    return np.array(fold_bacc), pred


def permute_labels(y, rng):
    # Xao theo benh nhan neu moi benh nhan chi co 1 nhan (giu cau truc nhieu mau/benh nhan); nguoc lai xao theo mau
    g = pd.Series(y).groupby(groups).nunique()
    if (g == 1).all():
        uniq = np.unique(groups)
        lab = pd.Series(y, index=groups).groupby(level=0).first().loc[uniq].to_numpy()
        mapping = dict(zip(uniq, rng.permutation(lab)))
        return np.array([mapping[gr] for gr in groups])
    return rng.permutation(y)

## 1. Nhãn thật

In [ ]:
real_rows, real_summary, real_genes = [], {}, {}
for t in TASKS:
    y = TASK_LABELS[t]
    t0 = time.perf_counter()
    fold_bacc, pred = run_cv(y)
    labels = np.unique(y)
    rec = recall_score(y, pred, labels=labels, average=None, zero_division=0)
    real_summary[t] = {"bacc_real": fold_bacc.mean(), "bacc_real_sd": fold_bacc.std(ddof=1),
                       "mcc_real_pooled": matthews_corrcoef(y, pred),
                       **{f"recall_{l}": r for l, r in zip(labels, rec)}}
    for i, b in enumerate(fold_bacc, 1):
        real_rows.append({"task": t, "fold": i, "bacc": b})
    cm = pd.DataFrame(confusion_matrix(y, pred, labels=labels),
                      index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
    cm.to_csv(os.path.join(RESULT_DIR, f"real_confusion_{t}.csv"))
    print(f"--- {t}: BAcc {fold_bacc.mean():.3f} ± {fold_bacc.std(ddof=1):.3f} "
          f"(muc ngau nhien {TASK_CHANCE[t]:.3f}) | MCC {real_summary[t]['mcc_real_pooled']:.3f} "
          f"| {time.perf_counter() - t0:.1f}s")
    print("recall theo lop:", dict(zip(labels.tolist(), np.round(rec, 3))))
    print(cm)

# Gen F-test duoc chon tren toan bo mau u (chi de tham khao, KHONG dung de danh gia)
top = ftest_top_k(X, y_stage, N_FEATURES)
print("\nTop 15 gene F-test (stage_all, toan bo mau u - chi tham khao):", gene_names[top[:15]].tolist())
pd.DataFrame(real_rows).to_csv(os.path.join(RESULT_DIR, "real_per_fold.csv"), index=False)

## 2. Permutation test (chạy lại toàn bộ pipeline trên nhãn đã xáo)

In [ ]:
from joblib import Parallel, delayed

N_WORKERS = os.cpu_count() if N_JOBS == -1 else N_JOBS


def chunks(ids):
    # Moi worker nhan 1 khoi -> X chi bi gui sang worker vai lan thay vi moi lan hoan vi
    return [c for c in np.array_split(np.asarray(ids), N_WORKERS) if len(c)]


def permutation_chunk(t, perm_ids):
    out = []
    for i in perm_ids:
        rng = np.random.default_rng([SEED, int(i)])
        fold_bacc, _ = run_cv(permute_labels(TASK_LABELS[t], rng))
        out.append({"task": t, "perm_id": int(i), "bacc": fold_bacc.mean()})
    return out


null_rows = []
for t in TASKS:
    t0 = time.perf_counter()
    res = Parallel(n_jobs=N_JOBS)(delayed(permutation_chunk)(t, c) for c in chunks(range(N_PERMUTATIONS)))
    null_rows += [r for part in res for r in part]
    print(f"{t}: {N_PERMUTATIONS} hoan vi xong trong {time.perf_counter() - t0:.0f}s")

null_df = pd.DataFrame(null_rows)
null_df.to_csv(os.path.join(RESULT_DIR, "null_scores.csv"), index=False)
null_df.groupby("task")["bacc"].describe().round(3)

## 3. Baseline random-k và Dummy

In [ ]:
def random_chunk(t, reps, model):
    out = []
    for rep in reps:
        rng = np.random.default_rng([SEED, 1, int(rep)])
        fold_bacc, _ = run_cv(TASK_LABELS[t], selector="random", rng=rng, model=model)
        out.append({"task": t, "baseline": "random_k" if model == "lr" else "dummy", "rep": int(rep),
                    "bacc": fold_bacc.mean()})
    return out


rand_rows = []
for t in TASKS:
    t0 = time.perf_counter()
    for model in ["lr", "dummy"]:
        res = Parallel(n_jobs=N_JOBS)(delayed(random_chunk)(t, c, model) for c in chunks(range(N_RANDOM_K)))
        rand_rows += [r for part in res for r in part]
    print(f"{t}: random-k + dummy xong trong {time.perf_counter() - t0:.0f}s")

rand_df = pd.DataFrame(rand_rows)
rand_df.to_csv(os.path.join(RESULT_DIR, "random_k_scores.csv"), index=False)
rand_df.groupby(["task", "baseline"])["bacc"].describe().round(3)

## Tổng hợp và kết luận RQ0

In [ ]:
rows = []
for t in TASKS:
    real = real_summary[t]["bacc_real"]
    null = null_df.loc[null_df.task == t, "bacc"].to_numpy()
    rk = rand_df.loc[(rand_df.task == t) & (rand_df.baseline == "random_k"), "bacc"].to_numpy()
    dm = rand_df.loc[(rand_df.task == t) & (rand_df.baseline == "dummy"), "bacc"].to_numpy()
    p_perm = (np.sum(null >= real) + 1) / (len(null) + 1)
    p_rand = (np.sum(rk >= real) + 1) / (len(rk) + 1)
    rows.append({
        "dataset": DATASET_NAME, "task": t, "n_samples": len(TASK_LABELS[t]), "n_features": N_FEATURES,
        "chance": TASK_CHANCE[t], **real_summary[t],
        "null_mean": null.mean(), "null_sd": null.std(ddof=1), "null_q95": np.quantile(null, 0.95),
        "p_perm": p_perm,
        "random_k_mean": rk.mean(), "random_k_q2.5": np.quantile(rk, 0.025), "random_k_q97.5": np.quantile(rk, 0.975),
        "p_vs_random_k": p_rand,
        "dummy_mean": dm.mean(),
        "gain_over_null": real - null.mean(), "gain_over_random_k": real - rk.mean(),
        "rq0_signal": p_perm < ALPHA, "rq0_beats_random_k": p_rand < ALPHA,
        "rq0_pass": (p_perm < ALPHA) and (p_rand < ALPHA),
    })

summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(RESULT_DIR, "summary.csv"), index=False)

for r in rows:
    verdict = ("DAT: co tin hieu va vuot random-k" if r["rq0_pass"] else
               "Co tin hieu nhung KHONG vuot random-k" if r["rq0_signal"] else
               "KHONG co bang chung ve tin hieu")
    print(f"{r['task']:15s} BAcc {r['bacc_real']:.3f} | null {r['null_mean']:.3f}±{r['null_sd']:.3f} "
          f"(p={r['p_perm']:.3f}) | random-k {r['random_k_mean']:.3f} (p={r['p_vs_random_k']:.3f}) -> {verdict}")

with open(os.path.join(RESULT_DIR, "config.json"), "w") as f:
    json.dump({"dataset": DATASET_NAME, "data_dir": DATA_DIR, "n_features": N_FEATURES, "n_splits": N_SPLITS,
               "n_permutations": N_PERMUTATIONS, "n_random_k": N_RANDOM_K, "seed": SEED, "lr_params": LR_PARAMS,
               "tasks": TASKS, "alpha": ALPHA, "n_tumour_samples": int(len(y_stage)),
               "n_patients_with_multiple_samples": n_dup, "selector": "f_classif top-k inside each fold",
               "scaling": "none (raw log2)", "cpu_count": os.cpu_count()}, f, indent=1)
summary.round(3).T

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(TASKS), figsize=(5 * len(TASKS), 3.6), squeeze=False)
for ax, t in zip(axes[0], TASKS):
    r = summary.set_index("task").loc[t]
    ax.hist(null_df.loc[null_df.task == t, "bacc"], bins=25, alpha=0.6, label="Null (nhãn xáo)")
    ax.hist(rand_df.loc[(rand_df.task == t) & (rand_df.baseline == "random_k"), "bacc"],
            bins=25, alpha=0.6, label="Random-k")
    ax.axvline(r["bacc_real"], color="k", lw=2, label=f"Nhãn thật {r['bacc_real']:.3f}")
    ax.axvline(r["chance"], color="gray", ls="--", lw=1, label="Mức ngẫu nhiên")
    ax.set_title(f"{DATASET_NAME} · {t}\np_perm={r['p_perm']:.3f}, p_vs_random={r['p_vs_random_k']:.3f}")
    ax.set_xlabel("BAcc (TB 5 fold)")
axes[0][0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "perm_test.png"), dpi=150)
plt.show()

In [ ]:
import shutil
zip_path = shutil.make_archive(f"/kaggle/working/perm_test_{DATASET_NAME}", "zip", RESULT_DIR)
print("Da nen:", zip_path)
for root, _, files in os.walk(RESULT_DIR):
    for fn in sorted(files):
        print(" ", os.path.relpath(os.path.join(root, fn), RESULT_DIR))